# Handcrafted Features and Failure Analysis

Objective: combine saved AIDE/SPAI inference outputs with streaming handcrafted
features to produce report-ready tables, plots, and six failure/interesting
cases for AI-generated image detection.

This notebook is presentation-only. Heavy image feature extraction is run by
`uv run python -m diffguard.cli.compute_handcrafted_features`, and gold
artifacts are built by `uv run python -m diffguard.cli.build_analysis_artifacts`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import SVG, Image, display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent.parent

SILVER_FEATURE_DIR = REPO_ROOT / "data" / "silver" / "handcrafted"
GOLD_DIR = REPO_ROOT / "data" / "gold" / "analysis"
PLOTS_DIR = GOLD_DIR / "plots"
FAILURE_CASE_DIR = GOLD_DIR / "failure_cases"

required_paths = [
    SILVER_FEATURE_DIR / "ntire_features.csv",
    SILVER_FEATURE_DIR / "z_image_turbo_features.csv",
    SILVER_FEATURE_DIR / "feature_config.json",
    GOLD_DIR / "merged_ntire.csv",
    GOLD_DIR / "merged_z_image_turbo.csv",
    GOLD_DIR / "metrics_comparison.csv",
    GOLD_DIR / "failure_cases.csv",
    PLOTS_DIR / "score_distribution.svg",
    PLOTS_DIR / "handcrafted_feature_boxplots.svg",
    PLOTS_DIR / "roc_comparison.svg",
    PLOTS_DIR / "tta_or_uncertainty_proxy_if_available.svg",
]
for artifact in required_paths:
    if not artifact.exists():
        raise FileNotFoundError(artifact)

print(f"Repo root: {REPO_ROOT}")
print(f"Silver handcrafted: {SILVER_FEATURE_DIR}")
print(f"Gold analysis: {GOLD_DIR}")

## Artifact Inventory

The silver layer contains per-image handcrafted features. The gold layer
contains merged analysis tables, aggregate metrics, final plots, and selected
case images.


In [ ]:
ntire = pd.read_csv(GOLD_DIR / "merged_ntire.csv")
zit = pd.read_csv(GOLD_DIR / "merged_z_image_turbo.csv")
metrics = pd.read_csv(GOLD_DIR / "metrics_comparison.csv")
failure_cases = pd.read_csv(GOLD_DIR / "failure_cases.csv")

inventory = pd.DataFrame(
    [
        {"artifact": "merged_ntire.csv", "rows": len(ntire), "columns": ntire.shape[1]},
        {
            "artifact": "merged_z_image_turbo.csv",
            "rows": len(zit),
            "columns": zit.shape[1],
        },
        {
            "artifact": "metrics_comparison.csv",
            "rows": len(metrics),
            "columns": metrics.shape[1],
        },
        {
            "artifact": "failure_cases.csv",
            "rows": len(failure_cases),
            "columns": failure_cases.shape[1],
        },
    ]
)
display(inventory)
display(ntire.head(3))

## Detector Metrics

On NTIRE, SPAI is stronger than AIDE on ROC-AUC/AP and threshold metrics in the
saved inference run. On Z-Image-Turbo, all samples are fake, so the useful
summary is score distribution and the rate above the 0.5 fake threshold.


In [ ]:
metric_columns = [
    "dataset",
    "detector",
    "roc_auc",
    "average_precision",
    "accuracy_at_0_5",
    "f1",
    "precision",
    "recall",
    "mean_score",
    "std_score",
    "median_score",
    "min_score",
    "max_score",
    "score_gt_0_5_rate",
]
existing_columns = [column for column in metric_columns if column in metrics.columns]
display(metrics[existing_columns].style.format(precision=4, na_rep=""))

## Score Distributions and ROC

The NTIRE real/fake score distributions overlap substantially, so threshold-only
use should be treated cautiously. The ROC comparison summarizes ranking
performance independent of the fixed 0.5 threshold.


In [ ]:
display(SVG(filename=PLOTS_DIR / "score_distribution.svg"))
display(SVG(filename=PLOTS_DIR / "roc_comparison.svg"))

## Handcrafted Feature Probe

Handcrafted descriptors capture entropy, smoothness/edge strength, brightness,
and DCT high-frequency energy. They are explanatory probes: useful for
diagnosing artifacts and failure modes, but not a replacement for deep
detectors.


In [ ]:
display(SVG(filename=PLOTS_DIR / "handcrafted_feature_boxplots.svg"))

feature_columns = [
    "entropy",
    "laplacian_variance",
    "dct_high_frequency_ratio",
    "brightness_mean",
    "brightness_std",
    "edge_density",
]
feature_summary = (
    pd.concat(
        [
            ntire.assign(group=ntire["label"].map({0: "NTIRE real", 1: "NTIRE fake"})),
            zit.assign(group="Z-Image-Turbo"),
        ],
        ignore_index=True,
    )
    .groupby("group")[feature_columns]
    .agg(["mean", "std", "median"])
)
display(feature_summary.style.format(precision=4))

## Feature and Score Correlation

Correlations are computed on the merged CSVs only. Strong correlations would
suggest that a detector is tracking simple artifacts; weak or moderate
correlations suggest the deep model uses additional cues.


In [ ]:
correlation_columns = feature_columns + ["aide_score", "spai_score"]
correlation = ntire[correlation_columns].corr(method="spearman")
score_correlation = correlation.loc[feature_columns, ["aide_score", "spai_score"]]
correlation_path = GOLD_DIR / "feature_score_spearman_correlation.csv"
score_correlation.to_csv(correlation_path)
display(score_correlation.style.format(precision=4))
print(f"Saved correlation table to {correlation_path}")

## Uncertainty Proxy

AIDE/SPAI disagreement and near-threshold mean score are practical triage
signals. In a fake-data prevention workflow, high-disagreement or low-margin
cases should be sent to human review rather than auto-accepted.


In [ ]:
display(SVG(filename=PLOTS_DIR / "tta_or_uncertainty_proxy_if_available.svg"))

## Failure Case Analysis

The six selected examples cover two confident true-positive fakes, two
missed/low-score fakes, one high-score real false positive, and one uncertain
or disagreement case. If strict criteria are not all available, the closest case
is selected and documented in `reason`.


In [ ]:
display(
    failure_cases[
        [
            "case_id",
            "case_type",
            "source",
            "label",
            "aide_score",
            "spai_score",
            "entropy",
            "laplacian_variance",
            "dct_high_frequency_ratio",
            "reason",
        ]
    ].style.format(precision=4)
)

In [ ]:
from PIL import Image as PILImage
from PIL import ImageDraw

thumbs = []
for row in failure_cases.itertuples(index=False):
    with PILImage.open(row.copied_path) as image:
        thumb = image.convert("RGB")
        thumb.thumbnail((220, 170))
        canvas = PILImage.new("RGB", (240, 230), "white")
        canvas.paste(thumb, ((240 - thumb.width) // 2, 8))
        draw = ImageDraw.Draw(canvas)
        label = f"{row.case_id} | {row.case_type}"
        scores = f"AIDE {row.aide_score:.2f}  SPAI {row.spai_score:.2f}"
        draw.text((10, 185), label[:32], fill="black")
        draw.text((10, 205), scores, fill="black")
        thumbs.append(canvas)

sheet = PILImage.new("RGB", (720, 460), "white")
for idx, thumb in enumerate(thumbs):
    x = (idx % 3) * 240
    y = (idx // 3) * 230
    sheet.paste(thumb, (x, y))
contact_sheet = PLOTS_DIR / "failure_cases_contact_sheet.png"
sheet.save(contact_sheet)
display(Image(filename=contact_sheet))
print(f"Saved contact sheet to {contact_sheet}")

## Interpretation for Report

SPAI is frequency-oriented and generalizes better on NTIRE in this run, while
AIDE is more sensitive to the Z-Image-Turbo probe set. Handcrafted entropy,
smoothness, DCT high-frequency ratio, brightness, and edge density do not
replace deep detectors, but they help explain smoothness and high-frequency
artifacts behind detector behavior. Disagreement and high-uncertainty cases
should be routed to human review in a system for preventing fake generated data
from entering trusted datasets.
